In [ ]:
import json
import os
from pathlib import Path

EMBEDDED_ABLATION_CONFIG = {}
WANDB_API_KEY = "__WANDB_API_KEY_PLACEHOLDER__"

CONFIG = {
    "repo_url": "https://github.com/sontungkieu/shortcut-models",
    "branch": "gmm",
    "dataset_ref": "codemaivanngu/shortcut-celebahq256",
    "dataset_name": "celebahq256",
    "tfds_data_dir": "/root/tensorflow_datasets",
    "dataset_download_dir": "/kaggle/working/shortcut_dataset",
    "batch_size": 64,
    "gmm_fit_samples": 32768,
    "gmm_valid_samples": 4096,
    "gmm_num_modes": 64,
    "gmm_em_iters": 25,
    "gmm_em_restarts": 1,
    "gmm_em_chunk_size": 128,
    "gmm_min_std": 0.0,
    "gmm_min_std_data_frac": 1.0,
    "gmm_pi_prior_type": "kl",
    "gmm_pi_prior_strength": 512.0,
    "gmm_pi_kl_steps": 100,
    "gmm_pi_kl_lr": 0.2,
    "wandb_project": "shortcut",
    "run_name": "gmm_ablation_manual",
    "jax_runtime": "cuda12",
}
CONFIG.update(EMBEDDED_ABLATION_CONFIG)
RUN_NAME = CONFIG["run_name"]

if WANDB_API_KEY and not WANDB_API_KEY.startswith("__"):
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
else:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB2")
    except Exception:
        pass

os.environ["MPLBACKEND"] = "agg"
os.environ["JAX_TRACEBACK_FILTERING"] = "off"
print(json.dumps(CONFIG, indent=2, sort_keys=True))


In [ ]:
!pip install -q kaggle tfds apache_beam mlcroissant
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] += ":/root/.local/bin"


In [ ]:
from pathlib import Path
import shutil

DATASET_REF = CONFIG["dataset_ref"]
DOWNLOAD_DIR = Path(CONFIG["dataset_download_dir"])
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
!kaggle datasets download -d {DATASET_REF} -p {str(DOWNLOAD_DIR)} --unzip

if (DOWNLOAD_DIR / "tensorflow_datasets").exists():
    target = Path("/root/tensorflow_datasets")
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(DOWNLOAD_DIR / "tensorflow_datasets", target)

%cd /kaggle/working
if not Path("tfds_builders").exists():
    !git clone https://github.com/kvfrans/tfds_builders.git
%cd tfds_builders/celebahq256
!tfds build


In [ ]:
from pathlib import Path
import shutil

%cd /kaggle/working
if not Path("shortcut-models").exists():
    !git clone {CONFIG["repo_url"]} shortcut-models
%cd shortcut-models
!git fetch --all
!git checkout {CONFIG["branch"]}
!git pull
!uv sync 1>sync_out.txt 2>sync_err.txt

if CONFIG.get("jax_runtime") == "cuda12":
    !uv pip install "jax[cuda12]==0.5.3" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html 1>jax_cuda_out.txt 2>jax_cuda_err.txt

source_data = Path(CONFIG["dataset_download_dir"]) / "data"
if source_data.exists():
    target_data = Path("/kaggle/working/shortcut-models/data")
    if target_data.exists():
        shutil.rmtree(target_data)
    shutil.copytree(source_data, target_data)


In [ ]:
import subprocess
from pathlib import Path

base_dir = Path("/kaggle/working/gmm_ablation") / RUN_NAME
diag_dir = base_dir / "diagnostics"
diag_dir.mkdir(parents=True, exist_ok=True)

gmm_stats_path = base_dir / "gmm_stats.npz"
prep_cmd = [
    "uv", "run", "data_prep.py",
    "--dataset_name", CONFIG["dataset_name"],
    "--tfds_data_dir", CONFIG["tfds_data_dir"],
    "--batch_size", str(CONFIG["batch_size"]),
    "--gmm_save_path", str(gmm_stats_path),
    "--gmm_latent_cache_path", str(base_dir / "gmm_latents.dat"),
    "--gmm_num_modes", str(CONFIG["gmm_num_modes"]),
    "--gmm_fit_samples", str(CONFIG["gmm_fit_samples"]),
    "--gmm_valid_samples", str(CONFIG["gmm_valid_samples"]),
    "--gmm_em_iters", str(CONFIG["gmm_em_iters"]),
    "--gmm_em_restarts", str(CONFIG["gmm_em_restarts"]),
    "--gmm_init_seed", "0",
    "--gmm_standardize_eps", "1e-6",
    "--gmm_pi_prior_type", CONFIG["gmm_pi_prior_type"],
    "--gmm_pi_prior_strength", str(CONFIG["gmm_pi_prior_strength"]),
    "--gmm_pi_kl_steps", str(CONFIG["gmm_pi_kl_steps"]),
    "--gmm_pi_kl_lr", str(CONFIG["gmm_pi_kl_lr"]),
    "--gmm_min_std", str(CONFIG["gmm_min_std"]),
    "--gmm_min_std_data_frac", str(CONFIG["gmm_min_std_data_frac"]),
    "--gmm_kmeanspp_init", "1",
    "--gmm_em_chunk_size", str(CONFIG["gmm_em_chunk_size"]),
    "--gmm_keep_latent_cache", "0",
    "--metrics_output_path", str(diag_dir / "gmm_metrics.json"),
    "--gmm_em_metrics_output_path", str(diag_dir / "gmm_em_metrics.jsonl"),
    "--wandb.name", f"prep_{RUN_NAME}",
]
with open(diag_dir / "gmm_prep_stdout.txt", "w") as out, open(diag_dir / "gmm_prep_stderr.txt", "w") as err:
    subprocess.run(prep_cmd, stdout=out, stderr=err, check=True)


In [ ]:
from pathlib import Path
base_dir = Path("/kaggle/working/gmm_ablation") / RUN_NAME
print("Ablation output:", base_dir)
!find {str(base_dir)} -maxdepth 3 -type f | sort
